## Импорт библиотек

In [1]:
import pandas as pd

import nltk
from nltk.corpus import stopwords

import pymorphy3

import re

import pickle

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from keras.src.utils import pad_sequences

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
nltk.download('averaged_perceptron_tagger_ru')

[nltk_data] Downloading package averaged_perceptron_tagger_ru to
[nltk_data]     C:\Users\robhul\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_ru is already up-to-
[nltk_data]       date!


True

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\robhul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
def preprocess_text(text):
    text = text.replace("ё", "е")
    text = re.sub(r'((www\.[^\s]+)|(https?://[^\s]+))', 'URL', text)
    text = re.sub('[^a-zA-Zа-яА-Я]+', ' ', text)
    text = re.sub(' +', ' ', text)
    text = text.lower()
    text = "".join(text)
    return text.strip()

In [5]:
data = []
with open('dataset.txt', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split(' ', 1)
        labels = parts[0].split(',')
        text = parts[1] if len(parts) > 1 else ""
        row = {
            'comment': preprocess_text(text),
            'normal':    1 if '__label__NORMAL' in labels else 0,
            'insult':    1 if '__label__INSULT' in labels else 0,
            'threat':    1 if '__label__THREAT' in labels else 0,
            'obscenity': 1 if '__label__OBSCENITY' in labels else 0
        }
        data.append(row)

df = pd.DataFrame(data)
df = df[df['comment'].str.strip() != '']

In [6]:
df.head(10)

,comment,normal,insult,threat,obscenity
0,скотина что сказать,0,1,0,0
1,я сегодня проезжала по рабочей и между домами ...,1,0,0,0
2,очередной лохотрон зачем придумывать очередной...,1,0,0,0
3,ретро дежавю сложно понять чужое сердце лиш ощ...,1,0,0,0
4,а когда мы статус агрогородка получили,1,0,0,0
5,августа поздно вечером нашли вот такую потеряш...,1,0,0,0
6,вчера надыбала новые стикеры u a ec fabs,1,0,0,0
7,заколоть этого плешивого урода что бы крякнул ...,0,1,1,0
8,а еще на стоянке никто не проверяет безопаснос...,1,0,0,0
9,красота если есть что показать почему нет,1,0,0,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 248290 entries, 0 to 248289
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   comment    248290 non-null  str  
 1   normal     248290 non-null  int64
 2   insult     248290 non-null  int64
 3   threat     248290 non-null  int64
 4   obscenity  248290 non-null  int64
dtypes: int64(4), str(1)
memory usage: 9.5 MB


In [8]:
stopwords = stopwords.words('russian')

In [9]:
def remove_stopwords(text):
    tokens = text.split()
    tokens = [word for word in tokens if word not in stopwords and len(word) > 2]
    return ' '.join(tokens)

In [10]:
df['prep_comment'] = df['comment'].apply(remove_stopwords)

In [11]:
morph = pymorphy3.MorphAnalyzer()

In [12]:
MAX_WORDS = 35000
MAX_LEN = 120

In [13]:
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['prep_comment'])

In [14]:
sequences = tokenizer.texts_to_sequences(df['prep_comment'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

In [15]:
df.head()

,comment,normal,insult,threat,obscenity,prep_comment
0,скотина что сказать,0,1,0,0,скотина сказать
1,я сегодня проезжала по рабочей и между домами ...,1,0,0,0,сегодня проезжала рабочей домами снитенко гомо...
2,очередной лохотрон зачем придумывать очередной...,1,0,0,0,очередной лохотрон придумывать очередной налог...
3,ретро дежавю сложно понять чужое сердце лиш ощ...,1,0,0,0,ретро дежавю сложно понять чужое сердце лиш ощ...
4,а когда мы статус агрогородка получили,1,0,0,0,статус агрогородка получили


In [16]:
labels = df[['normal','insult','threat','obscenity']].values

In [17]:
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.12, stratify=labels.argmax(axis=1))

In [18]:
model = Sequential([
    Embedding(MAX_WORDS, 128),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.35),
    Bidirectional(LSTM(32)),
    Dropout(0.35),
    Dense(64, activation='relu'),
    Dropout(0.25),
    Dense(4, activation='sigmoid')
])

In [19]:
model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy', 'Precision', 'Recall'])

In [20]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [21]:
history = model.fit(
    X_train, y_train,
    epochs=12,
    batch_size=96,
    validation_split=0.1,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/12
2276/2276 ━━━━━━━━━━━━━━━━━━━━ 357s 155ms/step - Precision: 0.9338 - Recall: 0.8925 - accuracy: 0.9255 - loss: 0.1302 - val_Precision: 0.9537 - val_Recall: 0.9374 - val_accuracy: 0.9482 - val_loss: 0.0838 - learning_rate: 0.0010
Epoch 2/12
2276/2276 ━━━━━━━━━━━━━━━━━━━━ 369s 162ms/step - Precision: 0.9624 - Recall: 0.9406 - accuracy: 0.9527 - loss: 0.0738 - val_Precision: 0.9598 - val_Recall: 0.9468 - val_accuracy: 0.9538 - val_loss: 0.0748 - learning_rate: 0.0010
Epoch 3/12
2276/2276 ━━━━━━━━━━━━━━━━━━━━ 406s 178ms/step - Precision: 0.9695 - Recall: 0.9606 - accuracy: 0.9614 - loss: 0.0536 - val_Precision: 0.9591 - val_Recall: 0.9487 - val_accuracy: 0.9529 - val_loss: 0.0800 - learning_rate: 0.0010
Epoch 4/12
2276/2276 ━━━━━━━━━━━━━━━━━━━━ 405s 178ms/step - Precision: 0.9768 - Recall: 0.9708 - accuracy: 0.9683 - loss: 0.0404 - val_Precision: 0.9578 - val_Recall: 0.9483 - val_accuracy: 0.9508 - val_loss: 0.0954 - learning_rate: 0.0010
Epoch 5/12
2276/2276 ━━━━━━━━━━━━━━━━━━━

In [25]:
model.save('multi_toxic_ru.h5')
with open('tokenizer_multi.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)